# Learned Image Compression: Rate-Distortion Focused Experiment Notebook

**Experiment plan**
- **A** Quantization method ablation (noise / STE / round)
- **B** Entropy model: factorized prior vs. scale hyperprior
- **C** Loss ablation: MSE vs MS-SSIM vs MSE+LPIPS
- **D** Baselines: JPEG, WebP, (optional) CompressAI pretrained models

> GPU (Colab T4 etc.) is recommended. The default `ITERS` is for a quick trial; try 50k-200k iterations for meaningful results.

## 0. Setup

In [ ]:
import os, io, math, glob, json, time, random, urllib.request
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from pytorch_msssim import ms_ssim

from compressai.entropy_models import EntropyBottleneck
from compressai.layers import GDN
from compressai.models import CompressionModel, ScaleHyperprior

# ---------------- Settings ----------------
SEED       = 42
TRAIN_DIR  = "./train_images"   
KODAK_DIR  = "./kodak"
CKPT_DIR   = "./ckpt"
RESULTS_JSON = "./results.json"
ITERS      = 10_000            
BATCH      = 16
PATCH      = 256
NUM_WORKERS = 0 if os.name == "nt" else 4  

LAMBDAS_MSE    = [0.0018, 0.0067, 0.025]    # PSNR-focused (using 255^2*MSE)
LAMBDAS_MSSSIM = [4.58, 16.64, 31.73]       # MS-SSIM-focused
QUANT_LAMBDAS  = [0.0067]                   # for Experiment A
USE_PRETRAINED_REF = False                   # CompressAI pretrained reference points

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)
os.makedirs(CKPT_DIR, exist_ok=True)
print("device:", device, "| torch", torch.__version__)

In [ ]:
import os, urllib.request, zipfile

url = "https://huggingface.co/datasets/yangtao9009/DIV2K/resolve/main/DIV2K_train.zip"
zip_path = "DIV2K_train_HR.zip"

if os.path.exists(zip_path):
    os.remove(zip_path)  # clean up old/partial file

print("Downloading (~3.5 GB, HF mirror, ~9 MB/s)")
urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path) as z:
    z.extractall("./train_images")
print("Done")


## 1. Data
- **Training:** random 256x256 crops from images in `TRAIN_DIR`. E.g. DIV2K train (800 images) or CLIC.
- **Test:** Kodak 24 images, **full resolution** (768x512), no resizing.

In [ ]:
IMG_EXT = (".png", ".jpg", ".jpeg")
def list_images(d):
    return sorted(p for p in glob.glob(os.path.join(d, "**", "*"), recursive=True)
                  if p.lower().endswith(IMG_EXT))

CACHE_MAX_SIDE = 768  # upper bound for the RAM cache (large enough relative to patch=256 to preserve crop diversity)

class PatchDataset(Dataset):
    def __init__(self, root, patch=256, cache_max_side=CACHE_MAX_SIDE):
        self.paths = list_images(root)
        assert len(self.paths) > 0, f"'{root}' içinde görüntü yok. TRAIN_DIR'i ayarla (örn. DIV2K)."
        self.tf = T.Compose([
            T.RandomCrop(patch, pad_if_needed=True, padding_mode="reflect"),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
        ])
        print(f"{len(self.paths)} görüntü RAM'e önbelleğe alınıyor (max kenar {cache_max_side}px)...")
        t0 = time.time()
        self.cache = []
        for i, p in enumerate(self.paths):
            im = Image.open(p).convert("RGB")
            w, h = im.size
            if max(w, h) > cache_max_side:
                scale = cache_max_side / max(w, h)
                im = im.resize((max(1, round(w * scale)), max(1, round(h * scale))), Image.BILINEAR)
            self.cache.append(im)
            if (i + 1) % 200 == 0:
                print(f"  {i+1}/{len(self.paths)}")
        print(f"Önbellekleme bitti: {time.time()-t0:.1f}s")

    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        return self.tf(self.cache[i])

def download_kodak(dest=KODAK_DIR):
    os.makedirs(dest, exist_ok=True)
    for i in range(1, 25):
        fn = f"kodim{i:02d}.png"; path = os.path.join(dest, fn)
        if not os.path.exists(path):
            urllib.request.urlretrieve(f"http://r0k.us/graphics/kodak/kodak/{fn}", path)
download_kodak()
kodak_paths = list_images(KODAK_DIR)
print("Kodak:", len(kodak_paths), "görüntü")

train_ds = PatchDataset(TRAIN_DIR, PATCH)
print("Train görüntü sayısı:", len(train_ds))

## 2. Metrics and real bpp
Instead of a compression ratio, we use **bpp = number of bitstream bits / (H*W)**. The bitstream is the actual byte sequence produced by the entropy coder.

In [ ]:
def psnr_t(a, b):
    mse = F.mse_loss(a, b).item()
    return 10 * math.log10(1.0 / max(mse, 1e-12))

def to_tensor_img(path):
    return TF.to_tensor(Image.open(path).convert("RGB")).unsqueeze(0)

def pad_to_multiple(x, m=64):
    h, w = x.shape[-2:]
    ph, pw = (m - h % m) % m, (m - w % m) % m
    return F.pad(x, (0, pw, 0, ph), mode="replicate"), (h, w)

@torch.no_grad()
def codec_roundtrip(model, x):
    """Real compress/decompress. Returns: rec (rounded to 8-bit), bpp"""
    xp, (h, w) = pad_to_multiple(x)
    out = model.compress(xp)
    bits = sum(len(s[0]) for s in out["strings"]) * 8
    rec = model.decompress(out["strings"], out["shape"])["x_hat"][..., :h, :w].clamp(0, 1)
    rec = (rec * 255).round() / 255
    return rec, bits / (h * w)

@torch.no_grad()
def evaluate_model(model, paths=None):
    paths = paths or kodak_paths
    model.eval(); model.update(force=True)
    rows = []
    for p in paths:
        x = to_tensor_img(p).to(device)
        rec, bpp = codec_roundtrip(model, x)
        rows.append((bpp, psnr_t(rec, x), ms_ssim(rec, x, data_range=1.0).item()))
    m = np.mean(rows, 0)
    return {"bpp": float(m[0]), "psnr": float(m[1]), "msssim": float(m[2])}

## 3. Models
**`SimpleCodec`**: 4-layer conv + GDN/IGDN, **factorized prior** entropy model. The `quant` parameter selects the quantization method used during training:
- `noise`: uniform noise (Balle et al.)
- `ste`: straight-through estimator
- `round`: plain `round` (no gradient, negative control)

**`ScaleHyperprior`**: CompressAI's hyperprior model (with the same N, M). This is the other end of the entropy-model ablation (B).

In [ ]:
class SimpleCodec(CompressionModel):
    def __init__(self, N=128, M=192, quant="noise"):
        super().__init__(entropy_bottleneck_channels=M)
        assert quant in ("noise", "ste", "round")
        self.quant = quant
        self.g_a = nn.Sequential(
            nn.Conv2d(3, N, 5, 2, 2), GDN(N),
            nn.Conv2d(N, N, 5, 2, 2), GDN(N),
            nn.Conv2d(N, N, 5, 2, 2), GDN(N),
            nn.Conv2d(N, M, 5, 2, 2),
        )
        self.g_s = nn.Sequential(
            nn.ConvTranspose2d(M, N, 5, 2, 2, output_padding=1), GDN(N, inverse=True),
            nn.ConvTranspose2d(N, N, 5, 2, 2, output_padding=1), GDN(N, inverse=True),
            nn.ConvTranspose2d(N, N, 5, 2, 2, output_padding=1), GDN(N, inverse=True),
            nn.ConvTranspose2d(N, 3, 5, 2, 2, output_padding=1),
        )

    def forward(self, x):
        y = self.g_a(x)
        y_eb, y_lik = self.entropy_bottleneck(y)   # train: with noise, eval: rounded
        if self.training and self.quant == "ste":
            y_hat = y + (torch.round(y) - y).detach()
        elif self.training and self.quant == "round":
            y_hat = torch.round(y)                 # no gradient: the encoder only learns via the rate term
        else:
            y_hat = y_eb
        return {"x_hat": self.g_s(y_hat), "likelihoods": {"y": y_lik}}

    def compress(self, x):
        y = self.g_a(x)
        return {"strings": [self.entropy_bottleneck.compress(y)], "shape": y.size()[-2:]}

    def decompress(self, strings, shape):
        y_hat = self.entropy_bottleneck.decompress(strings[0], shape)
        return {"x_hat": self.g_s(y_hat).clamp_(0, 1)}

def make_hyperprior():
    return ScaleHyperprior(N=128, M=192)

## 4. Loss: `L = R + λ·D`
`D` is selectable: `mse` (scaled by 255²), `msssim` (1 - MS-SSIM), or `mse_lpips`. Different losses need different λ ranges (lists above). `R` = the bpp estimate computed from the real likelihoods.

In [6]:
class RDLoss(nn.Module):
    def __init__(self, lmbda, metric="mse", lpips_w=0.01):
        super().__init__()
        self.lmbda, self.metric, self.lpips_w = lmbda, metric, lpips_w
        if metric == "mse_lpips":
            import lpips
            self.lp = lpips.LPIPS(net="alex", verbose=False)
            for p in self.lp.parameters(): p.requires_grad = False

    def forward(self, out, target):
        N, _, H, W = target.size()
        bpp = sum(torch.log(l).sum() for l in out["likelihoods"].values()) / (-math.log(2) * N * H * W)
        xh = out["x_hat"]
        mse = F.mse_loss(xh, target)
        if self.metric == "mse":
            dist = 255 ** 2 * mse
        elif self.metric == "msssim":
            dist = 1 - ms_ssim(xh, target, data_range=1.0)
        elif self.metric == "mse_lpips":
            dist = 255 ** 2 * (mse + self.lpips_w * self.lp(xh * 2 - 1, target * 2 - 1).mean())
        else:
            raise ValueError(self.metric)
        return {"loss": bpp + self.lmbda * dist, "bpp": bpp, "mse": mse}

## 5. Training loop
Two optimizers: the main network + `aux_loss` for the entropy bottleneck's `quantiles` parameters. Results are written to `results.json`; completed experiments are skipped (to survive Colab disconnects).

In [ ]:
def infinite(loader):
    while True:
        for b in loader:
            yield b

def train(model, loss_fn, iters, lr=1e-4, log_every=1000, tag=""):
    loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, drop_last=True,
                        num_workers=NUM_WORKERS, persistent_workers=NUM_WORKERS > 0)
    main_params = [p for n, p in model.named_parameters() if not n.endswith(".quantiles")]
    aux_params  = [p for n, p in model.named_parameters() if n.endswith(".quantiles")]
    opt, aux_opt = torch.optim.Adam(main_params, lr=lr), torch.optim.Adam(aux_params, lr=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=iters, eta_min=1e-6)
    model.train(); data = infinite(loader); t0 = time.time()
    for it in range(1, iters + 1):
        x = next(data).to(device)
        opt.zero_grad(); aux_opt.zero_grad()
        out = model(x); l = loss_fn(out, x)
        l["loss"].backward()
        torch.nn.utils.clip_grad_norm_(main_params, 1.0)
        opt.step()
        model.aux_loss().backward(); aux_opt.step(); sched.step()
        if it % log_every == 0 or it == iters:
            psnr = -10 * math.log10(l["mse"].item() + 1e-12)
            print(f"[{tag}] it {it}/{iters} | loss {l['loss'].item():.4f} | bpp {l['bpp'].item():.3f} | "
                  f"PSNR(train) {psnr:.2f} | {time.time()-t0:.0f}s")

RESULTS = json.load(open(RESULTS_JSON)) if os.path.exists(RESULTS_JSON) else {}
def safe(s): return s.replace("|", "_").replace(".", "p")

def run_experiment(exp, model_fn, metric, lmbdas, iters=ITERS):
    for lmbda in lmbdas:
        key = f"{exp}|{lmbda}"
        if key in RESULTS:
            print("atlandı (kayıtlı):", key); continue
        set_seed(SEED)
        model = model_fn().to(device)
        train(model, RDLoss(lmbda, metric).to(device), iters, tag=key)
        torch.save(model.state_dict(), f"{CKPT_DIR}/{safe(key)}.pth")  # save BEFORE update()
        r = evaluate_model(model)
        r.update(exp=exp, lmbda=lmbda, metric=metric)
        RESULTS[key] = r
        json.dump(RESULTS, open(RESULTS_JSON, "w"), indent=1)
        print(f">>> {key}: bpp {r['bpp']:.3f} | PSNR {r['psnr']:.2f} | MS-SSIM {r['msssim']:.4f}")

def by_exp(exp):
    return sorted([r for r in RESULTS.values() if r["exp"] == exp], key=lambda r: r["bpp"])

def by_exp_valid(exp):
    """by_exp + NaN-filtered: on Windows, GaussianConditional decode produces NaN for some λ values (native extension bug);
    we keep these points out of the plot/BD-rate calculations."""
    return [r for r in by_exp(exp) if not (math.isnan(r["psnr"]) or math.isnan(r["msssim"]))]

## 6. Experiments

In [ ]:
# --- A: quantization ablation (noise, run jointly with B_factorized) ---
run_experiment("B_factorized",   lambda: SimpleCodec(quant="noise"), "mse", sorted(set(QUANT_LAMBDAS + LAMBDAS_MSE)))
run_experiment("A_quant_ste",    lambda: SimpleCodec(quant="ste"),   "mse", QUANT_LAMBDAS)
run_experiment("A_quant_round",  lambda: SimpleCodec(quant="round"), "mse", QUANT_LAMBDAS)

In [ ]:
# --- B: entropy model ---
run_experiment("B_hyperprior", make_hyperprior, "mse", LAMBDAS_MSE)

In [ ]:
# --- C: loss ablation (same architecture: hyperprior) ---
run_experiment("C_hyper_msssim",     make_hyperprior, "msssim",    LAMBDAS_MSSSIM)
run_experiment("C_hyper_mse_lpips",  make_hyperprior, "mse_lpips", LAMBDAS_MSE)

## 7. Baselines: JPEG, WebP, (optional) pretrained CompressAI

In [8]:
def classic_baseline(fmt, qualities, paths=None):
    paths = paths or kodak_paths
    pts = []
    for q in qualities:
        rows = []
        for p in paths:
            img = Image.open(p).convert("RGB"); w, h = img.size
            buf = io.BytesIO(); img.save(buf, format=fmt, quality=q)
            data = buf.getvalue()
            rec = Image.open(io.BytesIO(data)).convert("RGB")
            x, y = TF.to_tensor(img)[None], TF.to_tensor(rec)[None]
            rows.append((len(data) * 8 / (w * h), psnr_t(y, x), ms_ssim(y, x, data_range=1.0).item()))
        m = np.mean(rows, 0)
        pts.append({"bpp": float(m[0]), "psnr": float(m[1]), "msssim": float(m[2]), "q": q})
    return sorted(pts, key=lambda r: r["bpp"])

JPEG = classic_baseline("JPEG", [5, 10, 15, 20, 30, 40, 50, 60, 75, 90])
WEBP = classic_baseline("WEBP", [5, 10, 20, 30, 40, 50, 60, 75, 90])

REF = []
if USE_PRETRAINED_REF:
    from compressai.zoo import bmshj2018_hyperprior
    for q in [1, 2, 3, 4, 5, 6]:
        m = bmshj2018_hyperprior(quality=q, pretrained=True).to(device)
        REF.append(evaluate_model(m))
    REF.sort(key=lambda r: r["bpp"])

## 8. RD curves and BD-rate

In [ ]:
def plot_rd(groups, metric="psnr", title="", xlim=None):
    plt.figure(figsize=(7, 5))
    for name, pts in groups.items():
        if not pts: continue
        x = [p["bpp"] for p in pts]
        y = [p[metric] if metric == "psnr" else -10 * math.log10(1 - p["msssim"] + 1e-12) for p in pts]
        plt.plot(x, y, marker="o", label=name)
    plt.xlabel("bpp (gerçek bitstream)")
    plt.ylabel("PSNR (dB)" if metric == "psnr" else "MS-SSIM (dB)  = -10·log10(1-MS-SSIM)")
    if xlim: plt.xlim(*xlim)
    plt.grid(alpha=.3); plt.legend(); plt.title(title); plt.show()

def bd_rate(anchor, test, metric="psnr"):
    """Bjontegaard delta-rate (%). Negative = test uses fewer bits at the same quality."""
    val = (lambda p: p["psnr"]) if metric == "psnr" else (lambda p: -10 * math.log10(1 - p["msssim"] + 1e-12))
    R1, D1 = np.log([p["bpp"] for p in anchor]), np.array([val(p) for p in anchor])
    R2, D2 = np.log([p["bpp"] for p in test]),   np.array([val(p) for p in test])
    lo, hi = max(D1.min(), D2.min()), min(D1.max(), D2.max())
    if hi <= lo: return float("nan")           # quality ranges don't overlap
    deg = min(3, len(D1) - 1, len(D2) - 1)
    p1, p2 = np.polyfit(D1, R1, deg), np.polyfit(D2, R2, deg)
    i1 = np.polyval(np.polyint(p1), hi) - np.polyval(np.polyint(p1), lo)
    i2 = np.polyval(np.polyint(p2), hi) - np.polyval(np.polyint(p2), lo)
    return (math.exp((i2 - i1) / (hi - lo)) - 1) * 100

In [ ]:
# Experiment A: quantization methods (same λ, single point)
A = {
    "noise": [r for r in by_exp("B_factorized") if r["lmbda"] in QUANT_LAMBDAS],
    "STE":   by_exp("A_quant_ste"),
    "round (gradyan yok)": by_exp("A_quant_round"),
}
for k, v in A.items():
    for r in v: print(f"{k:22s} bpp {r['bpp']:.3f}  PSNR {r['psnr']:.2f}  MS-SSIM {r['msssim']:.4f}")

In [ ]:
# Experiment B: entropy model + baselines
groups_B = {"JPEG": JPEG, "WebP": WEBP,
            "Factorized prior": by_exp("B_factorized"),
            "Scale hyperprior": by_exp_valid("B_hyperprior")}
if REF: groups_B["CompressAI pretrained (referans)"] = REF
plot_rd(groups_B, "psnr", "Kodak: PSNR - bpp", xlim=(0, 2.0))
plot_rd(groups_B, "msssim", "Kodak: MS-SSIM - bpp", xlim=(0, 2.0))

# NOTE: Only 1 λ (0.0018) gives a valid result for Scale hyperprior, the others are NaN
# (GaussianConditional.decode_with_indexes is broken in the locally-built 'ans' extension on Windows).
# It shows up as a single point on the plot; see the single-point comparison below for the full curve/BD-rate case.
b_valid = by_exp_valid("B_hyperprior")
f_match = [r for r in by_exp("B_factorized") if r["lmbda"] == 0.0018]
if b_valid and f_match:
    b0, f0 = b_valid[0], f_match[0]
    print(f"Factorized prior (λ=0.0018): bpp={f0['bpp']:.3f}  PSNR={f0['psnr']:.2f}")
    print(f"Scale hyperprior (λ=0.0018): bpp={b0['bpp']:.3f}  PSNR={b0['psnr']:.2f}")
    print(f"-> Aynı kalitede hyperprior ~%{100*(1-b0['bpp']/f0['bpp']):.0f} daha az bit kullanıyor")

In [ ]:
# Experiment C: loss ablation
groups_C = {"JPEG": JPEG,
            "hyperprior + MSE": by_exp_valid("B_hyperprior"),
            "hyperprior + MS-SSIM": by_exp_valid("C_hyper_msssim"),
            "hyperprior + MSE+LPIPS": by_exp_valid("C_hyper_mse_lpips")}
plot_rd(groups_C, "psnr", "Loss ablasyonu: PSNR", xlim=(0, 2.0))
plot_rd(groups_C, "msssim", "Loss ablasyonu: MS-SSIM", xlim=(0, 2.0))

# NOTE: hyperprior + MS-SSIM has no valid points at all (all NaN) - exclude this arm from the report.
# hyperprior + MSE+LPIPS also only has λ=0.0018 valid; single-point comparison against the MSE arm:
lp_valid = by_exp_valid("C_hyper_mse_lpips")
mse_valid = by_exp_valid("B_hyperprior")
if lp_valid and mse_valid:
    lp0, mse0 = lp_valid[0], mse_valid[0]
    print(f"hyperprior + MSE        (λ=0.0018): bpp={mse0['bpp']:.3f}  PSNR={mse0['psnr']:.2f}  MS-SSIM={mse0['msssim']:.4f}")
    print(f"hyperprior + MSE+LPIPS  (λ=0.0018): bpp={lp0['bpp']:.3f}  PSNR={lp0['psnr']:.2f}  MS-SSIM={lp0['msssim']:.4f}")

In [ ]:
# BD-rate table (JPEG anchor). Negative = bit savings relative to JPEG.
rows = []
for name, pts in {**{k: v for k, v in groups_B.items() if k not in ("JPEG",)},
                  **{k: v for k, v in groups_C.items() if k not in ("JPEG", "hyperprior + MSE")}}.items():
    if len(pts) >= 3:
        rows.append((name, bd_rate(JPEG, pts, "psnr"), bd_rate(JPEG, pts, "msssim")))
print(f"{'Model':38s} {'BD-rate PSNR':>14s} {'BD-rate MS-SSIM':>16s}")
for n, a, b in rows: print(f"{n:38s} {a:13.1f}% {b:15.1f}%")

## 9. Visual comparison (model vs JPEG at the same bpp)

In [ ]:
def load_model(model_fn, exp, lmbda):
    m = model_fn().to(device)
    m.load_state_dict(torch.load(f"{CKPT_DIR}/{safe(f'{exp}|{lmbda}')}.pth", map_location=device))
    m.eval(); m.update(force=True); return m

def jpeg_at_bpp(img, target_bpp):
    w, h = img.size; best = None
    for q in range(1, 96):
        buf = io.BytesIO(); img.save(buf, format="JPEG", quality=q)
        bpp = buf.tell() * 8 / (w * h)
        if best is None or abs(bpp - target_bpp) < abs(best[1] - target_bpp):
            best = (q, bpp, Image.open(io.BytesIO(buf.getvalue())).convert("RGB"))
    return best

def visual_compare(model, idx=0, crop=(200, 150, 328, 278)):
    p = kodak_paths[idx]; img = Image.open(p).convert("RGB"); x = to_tensor_img(p).to(device)
    rec, bpp = codec_roundtrip(model, x)
    q, jb, jimg = jpeg_at_bpp(img, bpp)
    jt = TF.to_tensor(jimg)[None].to(device)
    items = [("Orijinal", x[0].cpu(), ""),
             (f"Model  bpp={bpp:.3f}", rec[0].cpu(), f"PSNR {psnr_t(rec, x):.2f}"),
             (f"JPEG q={q}  bpp={jb:.3f}", jt[0].cpu(), f"PSNR {psnr_t(jt, x):.2f}")]
    l, t, r_, b = crop
    fig, ax = plt.subplots(2, 3, figsize=(13, 7))
    for i, (ttl, im, sub) in enumerate(items):
        ax[0, i].imshow(im.permute(1, 2, 0).clamp(0, 1)); ax[0, i].set_title(f"{ttl}\n{sub}"); ax[0, i].axis("off")
        ax[1, i].imshow(im[:, t:b, l:r_].permute(1, 2, 0).clamp(0, 1), interpolation="nearest"); ax[1, i].axis("off")
    plt.tight_layout(); plt.show()

# Example: the hyperprior model with the lowest λ (lowest bpp)
m = load_model(make_hyperprior, "B_hyperprior", LAMBDAS_MSE[0])
for i in (0, 7, 22): visual_compare(m, i)

## 10. Latent analysis: which channels are used?

In [ ]:
@torch.no_grad()
def channel_std(model, paths, n=6):
    model.eval(); ys = []
    for p in paths[:n]:
        x, _ = pad_to_multiple(to_tensor_img(p).to(device))
        ys.append(model.g_a(x).flatten(2))          # (1, M, HW)
    return torch.cat(ys, 2)[0].std(1).cpu().numpy()

for lm in LAMBDAS_MSE:
    s = channel_std(load_model(make_hyperprior, "B_hyperprior", lm), kodak_paths)
    plt.plot(np.sort(s)[::-1], label=f"λ={lm}  (aktif kanal ~{(s > 0.1 * s.max()).sum()}/{len(s)})")
plt.yscale("log"); plt.xlabel("kanal (sıralı)"); plt.ylabel("latent std")
plt.title("Düşük λ (düşük bpp) modeli daha az kanal kullanır"); plt.legend(); plt.show()

## 12. Validating our own arithmetic coder

Up to this point we've used CompressAI's `EntropyBottleneck` as a black box. In this section:

1. We write a standard **arithmetic coder** from scratch (the classic method that narrows the [0,1) interval according to probability, Witten-Neal-Cleary).
2. We extract a PMF for each channel from an already-trained model's (`B_factorized`) **learned probability distribution** (CompressAI's `_logits_cumulative`).
3. We use this PMF with our own coder to **actually** encode and decode the latent.
4. We check the result three ways: does the round-trip match exactly, how close is our bit count to the theoretical `Σ-log2p`, and how close is it to the real bit count produced by CompressAI's own `compress()`.

This is a check showing that we're not using CompressAI without understanding it - that we can independently write and validate a coder that works on the same principle.

In [ ]:
# --- Arithmetic coder (Witten-Neal-Cleary), from scratch ---
PREC = 32
TOP = (1 << PREC) - 1
HALF = 1 << (PREC - 1)
QUARTER = 1 << (PREC - 2)
AC_TOTAL_BITS = 16
AC_TOTAL = 1 << AC_TOTAL_BITS   # the sum of probability frequencies is normalized to this

def pmf_to_cdf(pmf):
    """Probabilities -> integer frequencies (sum=65536, each symbol >=1) -> cumulative table."""
    pmf = np.asarray(pmf, dtype=np.float64)
    pmf = pmf / pmf.sum()
    freq = np.maximum(1, np.round(pmf * AC_TOTAL).astype(np.int64))
    diff = AC_TOTAL - freq.sum()                  # load the rounding difference onto the largest symbol
    freq[np.argmax(freq)] += diff
    assert freq.min() >= 1 and freq.sum() == AC_TOTAL
    return np.concatenate([[0], np.cumsum(freq)]).tolist()

def ac_encode(symbols, cdf):
    """Encodes a list of symbols (indices between 0..len(cdf)-2) into a bit list."""
    low, high, pending, out = 0, TOP, 0, []
    def emit(bit):
        nonlocal pending
        out.append(bit); out.extend([1 - bit] * pending); pending = 0
    for s in symbols:
        rng = high - low + 1
        high = low + rng * cdf[s + 1] // AC_TOTAL - 1
        low  = low + rng * cdf[s]     // AC_TOTAL
        while True:
            if high < HALF:
                emit(0)
            elif low >= HALF:
                emit(1); low -= HALF; high -= HALF
            elif low >= QUARTER and high < HALF + QUARTER:
                pending += 1; low -= QUARTER; high -= QUARTER
            else:
                break
            low, high = 2 * low, 2 * high + 1
    pending += 1
    emit(0 if low < QUARTER else 1)
    return out

def ac_decode(bits, n, cdf):
    """Decodes n symbols back from the bit list produced by ac_encode."""
    it = iter(bits)
    nxt = lambda: next(it, 0)
    value = 0
    for _ in range(PREC):
        value = (value << 1) | nxt()
    low, high, res = 0, TOP, []
    for _ in range(n):
        rng = high - low + 1
        scaled = ((value - low + 1) * AC_TOTAL - 1) // rng
        s = int(np.searchsorted(cdf, scaled, side="right")) - 1
        res.append(s)
        high = low + rng * cdf[s + 1] // AC_TOTAL - 1
        low  = low + rng * cdf[s]     // AC_TOTAL
        while True:
            if high < HALF:
                pass
            elif low >= HALF:
                low -= HALF; high -= HALF; value -= HALF
            elif low >= QUARTER and high < HALF + QUARTER:
                low -= QUARTER; high -= QUARTER; value -= QUARTER
            else:
                break
            low, high = 2 * low, 2 * high + 1
            value = (value << 1) | nxt()
    return res

# Quick self-test: A=0.5, B=0.3, C=0.2 -> "BAC"
_cdf = pmf_to_cdf([0.5, 0.3, 0.2])
_syms = [1, 0, 2]  # B,A,C
_bits = ac_encode(_syms, _cdf)
_dec = ac_decode(_bits, len(_syms), _cdf)
print("oz-test round-trip:", _dec == _syms, "| bit sayisi:", len(_bits))

### 12.1 Validation on the real latent

We extract the probability table each channel learned from a trained `B_factorized` model's `entropy_bottleneck` (`_logits_cumulative`, the monotonic CDF network - the structure we described in section 3), and use it with our own coder.

`K` is how many integer values are included in the table (`-K..K`). If a channel's real values overflow this range (rare), that value is clipped to the nearest edge bin - a simplified version of a situation real coders solve with an "escape symbol". To choose `K` large enough, we first check `pmf.sum()`: it should come out very close to 1.

In [ ]:
@torch.no_grad()
def compressai_channel_table(eb, K=40):
    """Extracts, from a trained CompressAI EntropyBottleneck, the PMF over integer values
    and the learned median for each channel."""
    C = eb.channels
    dev = next(eb.parameters()).device
    medians = eb.quantiles[:, 0, 1]                      # (C,) learned median
    ks = torch.arange(-K, K + 1, dtype=torch.float32, device=dev)
    vals = medians.view(C, 1, 1) + ks.view(1, 1, -1)     # (C,1,2K+1)
    lower = eb._logits_cumulative(vals - 0.5, stop_gradient=True)
    upper = eb._logits_cumulative(vals + 0.5, stop_gradient=True)
    sign = -torch.sign(lower + upper)
    pmf = torch.abs(torch.sigmoid(sign * upper) - torch.sigmoid(sign * lower)).squeeze(1)
    return pmf.clamp_min(1e-9).cpu().numpy(), medians.cpu().numpy()


def validate_own_coder(model, img_path, K=40, max_channels=None):
    """model: a trained SimpleCodec (factorized). img_path: a Kodak image.
    Encodes with our own arithmetic coder and compares against CompressAI's real bit count.
    max_channels=None -> all channels (slow but directly comparable to CompressAI).
    For a quick sanity check you can pass e.g. 32 (in that case CompressAI's total covers all
    channels so it can't be compared directly - only look at the own/theoretical difference)."""
    model.eval()
    x = to_tensor_img(img_path).to(device)
    xp, (h, w) = pad_to_multiple(x)

    with torch.no_grad():
        y = model.g_a(xp)
    C = y.shape[1]
    ch_range = range(C) if max_channels is None else range(min(max_channels, C))

    pmf_table, medians = compressai_channel_table(model.entropy_bottleneck, K)
    coverage = pmf_table.sum(1).mean()
    if coverage < 0.99:
        print(f"  [uyari] ortalama pmf kapsama = {coverage:.4f} (<0.99) -> K'yi artir")

    total_own_bits, total_theo = 0, 0.0
    for c in ch_range:
        med = medians[c]
        symbols = (torch.round(y[0, c].flatten() - med).clamp(-K, K).long() + K).tolist()
        pmf = pmf_table[c]
        cdf = pmf_to_cdf(pmf)
        bits = ac_encode(symbols, cdf)
        dec = ac_decode(bits, len(symbols), cdf)
        assert dec == symbols, f"kanal {c}: round-trip basarisiz!"
        total_own_bits += len(bits)
        p = pmf / pmf.sum()
        total_theo += -np.log2(np.take(p, symbols)).sum()

    with torch.no_grad():
        out = model.compress(xp)
    compressai_bits = sum(len(s[0]) for s in out["strings"]) * 8
    n_used = len(list(ch_range))

    print(f"{os.path.basename(img_path)} | kullanilan kanal: {n_used}/{C} | pmf kapsama: {coverage:.4f}")
    print(f"  kendi kodlayicimiz      : {total_own_bits} bit")
    print(f"  teorik (-log2p toplami) : {total_theo:.0f} bit  (fark %{100*(total_own_bits/total_theo-1):.3f})")
    if n_used == C:
        print(f"  CompressAI compress()   : {compressai_bits} bit  (fark %{100*(total_own_bits/compressai_bits-1):.3f})")
    else:
        print(f"  CompressAI compress() (tum kanallar, kiyaslanamaz): {compressai_bits} bit")

In [ ]:
# Validation on a trained factorized model.
# max_channels=None (all channels) is directly comparable to CompressAI but slow;
# for a quick sanity check, first try max_channels=32.
val_model = load_model(lambda: SimpleCodec(quant="noise"), "B_factorized", QUANT_LAMBDAS[0])
for p in kodak_paths[:2]:
    validate_own_coder(val_model, p, K=40, max_channels=32)

### 12.2 End-to-end image with our own coder: encode → decode → pass through the decoder

The validation in 12.1 only checked the **bit count**, it didn't produce an image. Here we encode and decode all channels with our own coder, put the decoded integer symbols back into the latent's shape, and **pass them through `model.g_s` (the decoder)**. The result should come out **pixel-for-pixel identical** to the image produced by CompressAI's `decompress()`, because the same latent values go through the same decoder - the only difference is the entropy-coding back end.

This is visual proof of the claim that "the coder we wrote ourselves can really compress and decompress an image end to end".

In [ ]:
def own_decode_full_image(model, img_path, K=40):
    """Encodes/decodes all channels with our own arithmetic coder, passes them through the decoder
    to reconstruct the image. Compares against CompressAI's own decompress() output."""
    model.eval()
    x = to_tensor_img(img_path).to(device)
    xp, (h, w) = pad_to_multiple(x)
    with torch.no_grad():
        y = model.g_a(xp)
    B, C, Hy, Wy = y.shape
    pmf_table, medians = compressai_channel_table(model.entropy_bottleneck, K)

    y_hat = torch.zeros_like(y)
    total_bits = 0
    t0 = time.time()
    for c in range(C):
        med = medians[c]
        symbols = (torch.round(y[0, c].flatten() - med).clamp(-K, K).long() + K).tolist()
        pmf = pmf_table[c]
        cdf = pmf_to_cdf(pmf)
        bits = ac_encode(symbols, cdf)
        dec = ac_decode(bits, len(symbols), cdf)
        assert dec == symbols, f"kanal {c}: round-trip basarisiz!"
        total_bits += len(bits)
        vals = torch.tensor(dec, dtype=torch.float32, device=device) - K + med
        y_hat[0, c] = vals.view(Hy, Wy)
    print(f"kendi kodlayicimizla kodlama+cozme: {time.time()-t0:.1f}s")

    with torch.no_grad():
        rec_own = model.g_s(y_hat).clamp(0, 1)[..., :h, :w]
    rec_own = (rec_own * 255).round() / 255

    rec_compressai, bpp_compressai = codec_roundtrip(model, x)
    bpp_own = total_bits / (h * w)
    max_diff = (rec_own - rec_compressai).abs().max().item()

    print(f"bpp   kendi kodlayicimiz : {bpp_own:.4f}   | CompressAI: {bpp_compressai:.4f}")
    print(f"PSNR  kendi kodlayicimiz : {psnr_t(rec_own, x):.2f}  | CompressAI: {psnr_t(rec_compressai, x):.2f}")
    print(f"kendi vs CompressAI en buyuk piksel farki: {max_diff:.6f}  (0 ise piksel piksel ayni)")

    fig, ax = plt.subplots(1, 3, figsize=(12, 4.5))
    items = [("Orijinal", x[0].cpu()),
             (f"Kendi kodlayıcımız\nbpp={bpp_own:.3f}", rec_own[0].cpu()),
             (f"CompressAI\nbpp={bpp_compressai:.3f}", rec_compressai[0].cpu())]
    for a, (t, im) in zip(ax, items):
        a.imshow(im.permute(1, 2, 0).clamp(0, 1)); a.set_title(t); a.axis("off")
    plt.tight_layout(); plt.show()
    return rec_own, rec_compressai

# With all channels, on a single Kodak image (takes a few seconds)
_ = own_decode_full_image(val_model, kodak_paths[0], K=40)